# 02. File I/O, Serialization & Storage Formats: Beginner Guide

### 📌 Overview
Master **02. File I/O, Serialization & Storage Formats: Beginner Guide** with concise, zero-fluff bullet points and executable code on real Fintech records ([raw_transactions.csv](file:///data/raw_transactions.csv)).

### 📚 Key Concepts Covered in this Notebook:
- **Text Formats**: Covers `pd.read_csv()`, `pd.read_fwf()`, and `df.to_csv()`.
- **Excel Formats**: Covers `pd.read_excel()` and `df.to_excel()`.
- **Columnar & Binary Formats**: Covers `pd.read_parquet()`, `df.to_parquet()`, and `pd.read_feather()`.
- **Structured Web & DB Formats**: Covers `pd.read_json()`, `pd.read_html()`, and `pd.read_sql()`.


In [1]:
# Setup imports & dataset loading
import pandas as pd
import numpy as np
import sys
import time
import os
import sqlite3
import matplotlib.pyplot as plt

# Load raw transactions dataset
csv_path = 'data/raw_transactions.csv' if os.path.exists('data/raw_transactions.csv') else '../data/raw_transactions.csv'
df = pd.read_csv(csv_path)
print(f"Pandas Version: {pd.__version__}")
print(f"Loaded raw_transactions.csv: {df.shape[0]} rows, {df.shape[1]} columns")
print(df.head(2))

Pandas Version: 2.2.2
Loaded raw_transactions.csv: 15000 rows, 11 columns
  transaction_id customer_id merchant_id  transaction_amount card_type  \
0       TX109326      C55082       M3549              607.78      Visa   
1       TX106376      C76616       M3068             1819.11      Visa   

  transaction_status device_type  account_age_months     transaction_date  \
0           Reversed      Mobile                   8  2026-02-17 08:28:57   
1            Pending         POS                  28          03-Jan-2025   

  region  is_fraud  
0  North         0  
1   West         1  


### 🔹 Ingesting Delimited Text: `pd.read_csv()`
- **What it does:** Reads a comma-separated values (CSV) file into a Pandas DataFrame with automatic type inference and customizable parsing options.
- **Syntax:** `pd.read_csv()`
- **Key Note:** Specifying `usecols` and explicit `dtype` dictionaries allows reading multi-gigabyte CSV files into small RAM allocations.
- **Dataset Application & Code Demonstration:** Applies Ingesting Delimited Text on fintech records using columns `card_type`, `transaction_amount`, `transaction_id` to demonstrate real-world execution.


In [2]:
df_sub = pd.read_csv(csv_path, nrows=10)
print('Read CSV Head (10 rows):\n', df_sub[['transaction_id', 'transaction_amount', 'card_type']])

Read CSV Head (10 rows):
   transaction_id  transaction_amount card_type
0       TX109326              607.78      Visa
1       TX106376             1819.11      Visa
2       TX103301               64.08      Visa
3       TX110701             1025.73      Amex
4       TX103284              772.74  Discover
5       TX104210              198.47  Discover
6       TX106427              217.23  Discover
7       TX104105             1070.66      Amex
8       TX114893                 NaN  Discover
9       TX114584             1320.66  Discover


### 🔹 Ingesting Fixed-Width Formats: `pd.read_fwf()`
- **What it does:** Reads a table of fixed-width formatted lines into a DataFrame using explicit column widths or inferred column boundaries.
- **Syntax:** `pd.read_fwf()`
- **Key Note:** Fixed-width files are standard in legacy mainframe banking and ACH clearing systems; `colspecs` provides exact character slicing.
- **Dataset Application & Code Demonstration:** Executes Ingesting Fixed-Width Formats on the transaction DataFrame (`df`) to inspect and transform tabular features.


In [3]:
os.makedirs('scratch', exist_ok=True)
with open('scratch/fwf_demo.txt', 'w') as f:
    f.write('TX_1001   500.25    Visa      \nTX_1002   1200.00   MasterCard\n')
df_fwf = pd.read_fwf('scratch/fwf_demo.txt', widths=[10, 10, 12], names=['tx_id', 'amount', 'card'])
print('Parsed Fixed-Width Feed:\n', df_fwf)

Parsed Fixed-Width Feed:
      tx_id   amount        card
0  TX_1001   500.25        Visa
1  TX_1002  1200.00  MasterCard


### 🔹 Exporting Delimited Text: `DataFrame.to_csv()`
- **What it does:** Writes the DataFrame to a comma-separated values (CSV) file or text buffer.
- **Syntax:** `DataFrame.to_csv()`
- **Key Note:** Always set `index=False` when exporting tabular data unless the index carries meaningful business identifier data.
- **Dataset Application & Code Demonstration:** Executes Exporting Delimited Text on the transaction DataFrame (`df`) to inspect and transform tabular features.


In [4]:
df.head(100).to_csv('scratch/sample_tx.csv', index=False)
print('CSV Written. File size:', os.path.getsize('scratch/sample_tx.csv'), 'bytes')

CSV Written. File size: 7721 bytes


### 🔹 Excel Spreadsheet Serialization: `pd.read_excel()` & `DataFrame.to_excel()`
- **What it does:** Reads and writes Excel workbook sheets (`.xlsx`, `.xls`) to and from DataFrames.
- **Syntax:** `pd.read_excel()`
- **Key Note:** Reading Excel files is significantly slower than CSV or Parquet; use `usecols` and specific `sheet_name` to optimize speed.
- **Dataset Application & Code Demonstration:** Executes Excel Spreadsheet Serialization on the transaction DataFrame (`df`) to inspect and transform tabular features.


In [5]:
print('Excel I/O API: pd.read_excel(filepath, sheet_name=0) / df.to_excel(filepath, index=False)')

Excel I/O API: pd.read_excel(filepath, sheet_name=0) / df.to_excel(filepath, index=False)


### 🔹 Columnar Storage: `pd.read_parquet()` & `DataFrame.to_parquet()`
- **What it does:** Reads and writes Apache Parquet columnar binary files with high compression and native schema retention.
- **Syntax:** `pd.read_parquet()`
- **Key Note:** Parquet preserves exact data types (including datetimes and categoricals) and supports predicate pushdown for lightning-fast reads.
- **Dataset Application & Code Demonstration:** Applies Columnar Storage on fintech records using columns `is_fraud`, `transaction_amount`, `transaction_id` to demonstrate real-world execution.


In [6]:
df.to_parquet('scratch/raw_transactions.parquet', compression='snappy')
df_proj = pd.read_parquet('scratch/raw_transactions.parquet', columns=['transaction_id', 'transaction_amount', 'is_fraud'])
print('Parquet Column Projection Loaded (5 rows):\n', df_proj.head())

Parquet Column Projection Loaded (5 rows):
   transaction_id  transaction_amount  is_fraud
0       TX109326              607.78         0
1       TX106376             1819.11         1
2       TX103301               64.08         0
3       TX110701             1025.73         0
4       TX103284              772.74         0


### 🔹 Ultra-Fast Feather I/O: `pd.read_feather()` & `DataFrame.to_feather()`
- **What it does:** Reads and writes the Apache Arrow Feather binary format for ultra-fast ephemeral disk serialization.
- **Syntax:** `pd.read_feather()`
- **Key Note:** Feather maps memory directly from Arrow IPC format, making it the fastest disk read/write format for Python/R interoperability.
- **Dataset Application & Code Demonstration:** Executes Ultra-Fast Feather I/O on the transaction DataFrame (`df`) to inspect and transform tabular features.


In [7]:
df.reset_index(drop=True).to_feather('scratch/raw_transactions.feather')
df_feather = pd.read_feather('scratch/raw_transactions.feather')
print('Feather Loaded Rows:', len(df_feather))

Feather Loaded Rows: 15000


### 🔹 JSON Serialization: `pd.read_json()` & `DataFrame.to_json()`
- **What it does:** Converts a JSON string, file, or buffer into a Pandas DataFrame.
- **Syntax:** `pd.read_json()`
- **Key Note:** For large API streaming datasets, use `lines=True` (JSON Lines format) with `chunksize` to avoid loading massive JSON objects into memory at once.
- **Dataset Application & Code Demonstration:** Applies JSON Serialization on fintech records using columns `card_type`, `transaction_amount`, `transaction_id` to demonstrate real-world execution.


In [8]:
json_data = df[['transaction_id', 'transaction_amount', 'card_type']].head(3).to_json(orient='records')
df_json = pd.read_json(json_data, orient='records')
print('Parsed JSON Records:\n', df_json)

Parsed JSON Records:
   transaction_id  transaction_amount card_type
0       TX109326              607.78      Visa
1       TX106376             1819.11      Visa
2       TX103301               64.08      Visa


C:\Users\DELL\AppData\Local\Temp\ipykernel_4748\2875197155.py:2: FutureWarning: Passing literal json to 'read_json' is deprecated and will be removed in a future version. To read from a literal string, wrap it in a 'StringIO' object.
  df_json = pd.read_json(json_data, orient='records')


### 🔹 HTML Table Extraction: `pd.read_html()`
- **What it does:** Scrapes and parses HTML tables from a URL, HTML file, or string, returning a list of DataFrames.
- **Syntax:** `pd.read_html()`
- **Key Note:** `pd.read_html()` returns a **list** of DataFrames (one per HTML table found), not a single DataFrame.
- **Dataset Application & Code Demonstration:** Demonstrates HTML Table Extraction with practical fintech data structures and variables in the following code block.


In [9]:
html_table = '<table><tr><th>Region</th><th>Total</th></tr><tr><td>North America</td><td>150000</td></tr></table>'
print('Scraped HTML Table:\n', pd.read_html(html_table)[0])

Scraped HTML Table:
           Region   Total
0  North America  150000


C:\Users\DELL\AppData\Local\Temp\ipykernel_4748\3577034756.py:2: FutureWarning: Passing literal html to 'read_html' is deprecated and will be removed in a future version. To read from a literal string, wrap it in a 'StringIO' object.
  print('Scraped HTML Table:\n', pd.read_html(html_table)[0])


### 🔹 SQL Database Ingestion: `pd.read_sql()`
- **What it does:** Base object if memory is from some other object (view), or None if array owns its memory buffer.
- **Syntax:** `ndarray.base`
  - **Parameters:**
    - `ndarray`: Target array.
- **Key Note:** If `arr.base is not None`, modifying `arr` mutates the underlying original array.
- **Dataset Application & Code Demonstration:** Executes SQL Database Ingestion on the transaction DataFrame (`df`) to inspect and transform tabular features.


In [10]:
conn = sqlite3.connect(':memory:')
df.head(1000).to_sql('transactions', conn, index=False)
sql_df = pd.read_sql('SELECT card_type, COUNT(*) as tx_count, AVG(transaction_amount) as avg_amt FROM transactions GROUP BY card_type', conn)
print('SQL Aggregated Results:\n', sql_df)

SQL Aggregated Results:
     card_type  tx_count      avg_amt
0        Amex       251  1009.648625
1    Discover       243   990.743734
2  MasterCard       243  1051.757478
3        Visa       263  1019.530121


## 💡 Real-World Practice & Scenarios
Practical scenarios and common data engineering questions explained with real examples.


### 🔍 Scenario: Q1: Parquet vs CSV Compression & Predicate Pushdown
- **Objective:** Q1: Parquet vs CSV Compression & Predicate Pushdown
- **Approach:** Compare the file size and query speed between `raw_transactions.csv` and `raw_transactions.parquet`.
- **Syntax:** `os.path.getsize('file.parquet')` vs `os.path.getsize('file.csv')`

In [11]:
csv_sz = os.path.getsize(csv_path)
pq_sz = os.path.getsize('scratch/raw_transactions.parquet')
print(f'CSV File Size: {csv_sz / 1024:.1f} KB')
print(f'Parquet File Size (Snappy): {pq_sz / 1024:.1f} KB ({(1 - pq_sz/csv_sz)*100:.1f}% reduction)')

CSV File Size: 1107.2 KB
Parquet File Size (Snappy): 344.1 KB (68.9% reduction)
